# Pairwise Quality Assessment - Attribute Analysis

Analyze how each method performs on different user-labeled quality attributes.

In [ ]:
# Imports
%matplotlib inline
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))
sys.path.insert(0, str(Path.cwd()))

from sim_bench.analysis.utils import get_project_root
from merge_pairwise_results import load_pairwise_results, merge_pairwise_results
from analyze_pairwise_by_attributes import (
    merge_results_with_labels,
    compute_accuracy_by_attribute,
    compute_accuracy_by_label_presence,
    plot_accuracy_by_majority_label,
    plot_method_performance_by_label,
)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print("✓ Imports successful")

## 1. Load Data

In [ ]:
# Configuration
PROJECT_ROOT = get_project_root()
# BENCHMARK_DIR = PROJECT_ROOT / "outputs" / "pairwise_benchmark_3hour" / "pairwise_20251120_100520"
BENCHMARK_DIR = PROJECT_ROOT / "outputs" / "pairwise_benchmark_test" / "pairwise_20251125_004908"
# MERGED_RESULTS_PATH = BENCHMARK_DIR / "merged_pairwise_results.csv"
# LABELS_PATH = Path(r"D:\Similar Images\automatic_triage_photo_series\photo_triage_pairs_keyword_labels.csv") # Labels from key word matching
LABELS_PATH = Path(r"D:\Similar Images\automatic_triage_photo_series\reviews_df_labeled_semantic_v4.csv") # Using chatgpt
METHODS_SUMMARY = BENCHMARK_DIR / "methods_summary.csv"
print(f"Benchmark dir: {BENCHMARK_DIR}")
print(f"Results file exists: {MERGED_RESULTS_PATH.exists()}")
print(f"Labels file exists: {LABELS_PATH.exists()}")

In [ ]:
results_dict = load_pairwise_results(BENCHMARK_DIR)

pairwise_merged = merge_pairwise_results(results_dict, include_scores=True, include_predictions=True)

In [ ]:
# Load pairwise results
# pairwise_merged = pd.read_csv(MERGED_RESULTS_PATH)
print(f"Loaded pairwise results: {pairwise_merged.shape}")
print(f"Columns: {list(pairwise_merged.columns[:10])}...")
pairwise_merged.head()

In [ ]:
df_methods_summary = pd.read_csv(METHODS_SUMMARY)

In [ ]:
df_labels = pd.read_csv(LABELS_PATH)
print(df_labels.columns)
df_labels.head()

In [ ]:
df_labels[['compareFile1', 'compareFile2', 'majority_label']]

In [ ]:
pairwise_merged.columns

In [ ]:
pairwise_merged[['series_id', 'pair_id', 'compareID1', 'compareID2']].head(10)

In [ ]:
pairwise_merged[['compareID1', 'compareID2']] = pairwise_merged['pair_id'].str.split('_', expand=True)[[1, 2]]
pairwise_merged[['compareID1', 'compareID2']] = pairwise_merged[['compareID1', 'compareID2']].astype(int)

In [ ]:
df_full_labels = pd.read_csv(r'D:\Similar Images\automatic_triage_photo_series\train_val\labels_db.csv')
print(df_full_labels.columns)

df_full_labels.head()

In [ ]:
df_labels.dtypes

In [ ]:
df_labels_max_vote = df_labels.groupby(['series_id', 'compareID1', 'compareID2'])['label'].agg(
    majority_label=lambda x: x.value_counts().idxmax(),
    vote_count=lambda x: x.value_counts().max()
).reset_index()

In [ ]:
combined_df = pairwise_merged.merge(df_labels_max_vote, on=['series_id',  'compareID1', 'compareID2'], how='left')


In [ ]:
df_labels_max_vote.head()

In [ ]:
len(pairwise_merged), len(combined_df)

In [ ]:
combined_df.columns

In [ ]:
# Merge with user attribute labels

# Get method names
pairwise_methods = [col.replace('_correct', '') for col in combined_df.columns if col.endswith('_correct')]

print(f"\nMerged data shape: {combined_df.shape}")
print(f"Pairs with labels: {len(combined_df)} ({len(combined_df)/len(pairwise_merged)*100:.1f}% coverage)")
print(f"\nMethods: {', '.join(pairwise_methods)}")

## 2. Accuracy by Majority Label

Show how each method performs on different attribute categories (composition, sharpness, exposure, etc.)

In [ ]:
for method in pairwise_methods:
    combined_df[method+'_max'] = combined_df[[method+'_score_a', method+'_score_b']].max(axis=1)
    combined_df[method+'_min'] = combined_df[[method+'_score_a', method+'_score_b']].min(axis=1)
    combined_df[method+'_log_ratio'] = np.log10(combined_df[method+'_max'] / combined_df[method+'_min'])
    combined_df[method+'_diff'] = combined_df[method+'_max'] - combined_df[method+'_min']

In [ ]:
summary_table = pd.DataFrame()

for method in pairwise_methods:
    mean_ratios = combined_df.groupby([method+'_correct'])[method+'_diff'].mean()
    summary_table[method] = mean_ratios

summary_table.index.name = 'correct'
summary_table.reset_index(inplace=True)
summary_table.T

In [ ]:
summary_table = pd.DataFrame()

for method in pairwise_methods:
    mean_ratios = combined_df.groupby([method+'_correct'])[method+'_ratio'].mean()
    summary_table[method] = mean_ratios

summary_table.index.name = 'correct'
summary_table.reset_index(inplace=True)
summary_table.T

In [ ]:
list(combined_df.columns)[-40:]

In [ ]:
combined_df[method+'_correct'].astype(str)

In [ ]:
import fivecentplots as fcp

In [ ]:
for method in ['CLIP-Aesthetic-Overall', 'Sharpness']:
    combined_df[method+'_correct_'] = np.where(combined_df[method+'_correct'] == 1, 'TRUE', 'FALSE')
    g1 = fcp.boxplot(df=combined_df, y=method+'_diff', groups=['majority_label', method+'_correct_'],
                title=method, legend=method+'_correct_')

    g1 = fcp.boxplot(df=combined_df, y=method+'_log_ratio', groups=['majority_label', method+'_correct_'],
                title=method, legend=method+'_correct_')
    # g2 = sns.relplot(data=combined_df, x=method+'_min', y=method+'_max', hue=method+'_correct')
    # g2.fig.suptitle(method)
    # g2.fig.tight_layout()

In [ ]:
# Compute accuracy by majority label
majority_label_accuracy = compute_accuracy_by_attribute(combined_df, pairwise_methods, 'majority_label')

print("Method Accuracy by User-Labeled Attribute:")
print("="*80)
display(majority_label_accuracy.sort_values('count', ascending=False))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=df_methods_summary, x='method', y='avg_top1_accuracy', ax=ax)

# Add value labels
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f')

plt.xlabel('Method')
plt.ylabel('Average Top-1 Accuracy')
plt.title('Average Top-1 Accuracy by Method')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 3. Attribute Presence Analysis

Compare performance when specific attributes are mentioned vs not mentioned

In [ ]:
# Performance when specific attributes mentioned
label_cols = [col for col in combined_df.columns if col.startswith('label_') and 
              col not in ['label_no_reason_given', 'label_no_specific_attribute']]

label_presence = compute_accuracy_by_label_presence(combined_df, pairwise_methods, label_cols)

print("Accuracy when attribute PRESENT vs ABSENT:")
print("="*80)
display(label_presence.sort_values('present_count', ascending=False))

## 4. Composition Deep-Dive

In [ ]:
# Composition analysis
composition_pairs = combined_df[combined_df['label_composition'] > 0]
print(f"COMPOSITION ANALYSIS ({len(composition_pairs)} pairs where 'composition' mentioned)")
print("="*80)

for method in pairwise_methods:
    correct_col = f'{method}_correct'
    comp_acc = composition_pairs[correct_col].mean()
    overall_acc = combined_df[correct_col].mean()
    diff = comp_acc - overall_acc
    
    sign = "✓" if diff > 0 else "✗" if diff < 0 else "="
    print(f"{sign} {method:25s}: {comp_acc:.3f} (overall: {overall_acc:.3f}, diff: {diff:+.3f})")

## 5. Per-Method Visualizations

Detailed analysis for top methods

In [ ]:
# Visualize top 3 methods
for method in ['Combined-RuleBased', 'Sharpness', 'Colorfulness']:
    print(f"\n{'='*80}")
    print(f"{method} - Performance by Attribute")
    print(f"{'='*80}")
    fig = plot_method_performance_by_label(label_presence, method, figsize=(14, 6))
    if fig:
        plt.show()
    else:
        print(f"Could not create visualization for {method}")